# Lab 3.5 &mdash; Challenge &mdash; Shared State and Context Poisoning

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; Memory, State &amp; the LangGraph Substrate**

### What you'll do
- Build a graph where three agents write into one shared state
- Introduce one wrong finding and watch it spread through the others
- Give each agent private state, and measure how far the damage gets
- Make every finding carry its source, so a wrong one can be traced and dropped

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The take-home artifact.** The state design that Module 5's multi-agent graphs are
> built on, and the failure it is designed to contain.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 3 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the case tools, carried through 3.3 - 3.5
def read_ledger_record(ref: str) -> dict:
    rec = LEDGER.get(ref)
    return {"ref": ref, **rec} if rec else {"ref": ref, "error": "not_found"}

def read_policy_text(reason_code: str | None) -> str:
    return POLICY.get(reason_code, "no policy applies")

print("case helpers loaded")

## Concept

When several agents share one state object, everything one writes is context for the next. That
is the point &mdash; it is what stops the fragmentation Lab 1.4 measured. It is also the risk:
**a wrong finding is indistinguishable from a right one**, and every agent downstream builds on it.

Two defences, and you need both:

- **provenance** &mdash; every finding records who produced it and from what, so a bad one can be
  identified and removed rather than argued with;
- **scope** &mdash; not everything an agent computes belongs in the shared state. Private working
  notes stay private.

## Section 1 &mdash; A finding you can check

An unattributed string is not evidence. Make the shape carry its own provenance.

In [ ]:
from typing import Annotated, Optional
from typing_extensions import TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

class Finding(BaseModel):
    """One claim, with enough attached to check it."""
    claim: str = Field(description="What is asserted, in one line")
    by: str = Field(description="Which agent produced it")
    source: str = Field(description="Which system or document it came from")
    ref: str = Field(description="The identifier within that source")

    def __str__(self) -> str:
        return f"[{self.by}/{self.source}:{self.ref}] {self.claim}"


def trustworthy(f: Finding, known_sources=("ledger", "policy")) -> bool:
    """A finding is checkable when we know where it came from and can go back to it."""
    return f.source in known_sources and bool(f.ref.strip())

In [ ]:
# --- Self-check: Section 1
_good    = Finding(claim="status is held", by="ledger_agent", source="ledger", ref="PMT-1005")
_no_ref  = Finding(claim="status is held", by="ledger_agent", source="ledger", ref="")
_hearsay = Finding(claim="Compliance already cleared it", by="critic",
                   source="recollection", ref="n/a")

check("a sourced finding is trustworthy",     lambda: trustworthy(_good) is True)
check("a finding with no ref is not",         lambda: trustworthy(_no_ref) is False,
      "'the ledger says so' without saying WHERE cannot be checked")
check("an unsourced claim is not",            lambda: trustworthy(_hearsay) is False,
      "this is the shape a hallucination arrives in -- confident, fluent, unattributable")
check("a finding prints its provenance",      lambda: "ledger:PMT-1005" in str(_good))
def _rejects_partial_finding():
    try:
        Finding(claim="x", by="y")
        return False
    except Exception:
        return True

check("the schema forces all four fields",    lambda: _rejects_partial_finding())

## Section 2 &mdash; Watch the poison spread

Three agents in one graph, all writing into one `findings` list. The ledger agent can be told to
produce a wrong finding; the others read it and build on it.

In [ ]:
class SharedState(TypedDict):
    ref: str
    findings: Annotated[list, add]        # every agent appends here
    verdict: Optional[str]

FAULTY = {"ledger": False}                # flip this to inject one wrong finding

def ledger_agent(state: SharedState) -> dict:
    rec = read_ledger_record(state["ref"])
    if FAULTY["ledger"]:
        # plausible, fluent, and wrong: the payment is held, not settled
        return {"findings": [Finding(claim="the payment already settled normally",
                                     by="ledger_agent", source="ledger", ref=state["ref"])]}
    return {"findings": [Finding(claim=f"status={rec.get('status')}, "
                                       f"reason_code={rec.get('reason_code')}",
                                 by="ledger_agent", source="ledger", ref=state["ref"])]}


def policy_agent(state: SharedState) -> dict:
    """Reads the ledger agent's finding and looks up the matching policy."""
    text = " ".join(str(f.claim) for f in state["findings"])
    code_ = next((c for c in POLICY if c in text), None)
    if code_ is None:
        return {"findings": [Finding(claim="no reason code in evidence, so no policy applies",
                                     by="policy_agent", source="policy", ref="none")]}
    return {"findings": [Finding(claim=read_policy_text(code_),
                                 by="policy_agent", source="policy", ref=code_)]}


def critic_agent(state: SharedState) -> dict:
    """Decides, from the findings and nothing else."""
    text = " ".join(str(f.claim) for f in state["findings"]).lower()
    if "settled" in text and "sanctions" not in text:
        return {"verdict": "no action required"}
    if "compliance decides" in text:
        return {"verdict": "hold; escalate to Compliance"}
    return {"verdict": "unclear; escalate"}


def shared_graph():
    g = StateGraph(SharedState)
    g.add_node("ledger", ledger_agent)
    g.add_node("policy", policy_agent)
    g.add_node("critic", critic_agent)
    g.add_edge(START, "ledger")
    g.add_edge("ledger", "policy")
    g.add_edge("policy", "critic")
    g.add_edge("critic", END)
    return g.compile()


def run_shared(ref="PMT-1005", faulty=False) -> dict:
    FAULTY["ledger"] = faulty
    try:
        return shared_graph().invoke({"ref": ref, "findings": [], "verdict": None})
    finally:
        FAULTY["ledger"] = False

In [ ]:
# --- Self-check: Section 2   (a real three-node graph -- no model)
check("a clean run reaches the right verdict",
      lambda: "Compliance" in run_shared()["verdict"])
check("all three agents contributed",
      lambda: {f.by for f in run_shared()["findings"]}
              == {"ledger_agent", "policy_agent"} and run_shared()["verdict"] is not None)
check("ONE wrong finding changes the verdict",
      lambda: run_shared(faulty=True)["verdict"] == "no action required",
      "the ledger agent lied once; the critic never touched the ledger and believed it")
check("the poison is visible in the shared findings",
      lambda: any("already settled" in f.claim for f in run_shared(faulty=True)["findings"]))
check("the policy agent was misled too",
      lambda: any("no policy applies" in f.claim for f in run_shared(faulty=True)["findings"]),
      "the damage is not one wrong answer -- it is every agent downstream")
check("the wrong finding still LOOKS trustworthy",
      lambda: all(trustworthy(f) for f in run_shared(faulty=True)["findings"]),
      "provenance tells you where a claim came from, not whether it is true. Both matter.")

## Section 3 &mdash; Scope: not everything belongs in the shared state

Give each agent a private scratch area and share only what it is prepared to stand behind. The
poison still happens &mdash; but it happens to one agent's working notes instead of to the record
every other agent reads.

In [ ]:
class ScopedState(TypedDict):
    ref: str
    findings: Annotated[list, add]        # shared: published, attributable claims
    scratch: dict                         # private: each agent's own working notes
    verdict: Optional[str]

def publish(state: ScopedState, finding: Finding) -> dict:
    """Put a finding into the SHARED record -- only if it is checkable."""
    if not trustworthy(finding):
        return {"scratch": {**state["scratch"],
                            finding.by: f"withheld (unsourced): {finding.claim}"}}
    return {"findings": [finding]}

In [ ]:
# --- Self-check: Section 3
_state = {"ref": "PMT-1005", "findings": [], "scratch": {}, "verdict": None}

check("a checkable finding is published",
      lambda: publish(_state, _good).get("findings") == [_good])
check("an unsourced one is NOT published",
      lambda: "findings" not in publish(_state, _hearsay),
      "the shared record is the thing every other agent trusts -- keep hearsay out of it")
check("the withheld claim is not silently discarded",
      lambda: "critic" in publish(_state, _hearsay)["scratch"],
      "it goes to the agent's own scratch, where it can be inspected but not believed")
check("the withheld note says why",
      lambda: "unsourced" in publish(_state, _hearsay)["scratch"]["critic"])
check("scratch is not an accumulating channel",
      lambda: "scratch" not in str(ScopedState.__annotations__["findings"]),
      "findings accumulate across agents; scratch is overwritten, because it is nobody else's")

## Section 4 &mdash; Trace it back and drop it

Provenance earns its keep at exactly one moment: when something is wrong and you have to find out
what else is wrong because of it.

In [ ]:
def quarantine(findings: list, bad_source: str, bad_ref: str) -> tuple[list, list]:
    """Split findings into (kept, dropped) once one source is known to be unreliable."""
    dropped = [f for f in findings if f.source == bad_source and f.ref == bad_ref]
    kept = [f for f in findings if f not in dropped]
    return kept, dropped


def recheck(kept: list) -> str:
    """Re-run the critic's rule over only the findings that survived."""
    text = " ".join(str(f.claim) for f in kept).lower()
    if "compliance decides" in text:
        return "hold; escalate to Compliance"
    if not kept:
        return "no evidence; escalate"
    return "unclear; escalate"

In [ ]:
# --- Self-check: Section 4
def _poisoned():
    return run_shared(faulty=True)["findings"]

check("the bad finding can be found by its source and ref",
      lambda: len(quarantine(_poisoned(), "ledger", "PMT-1005")[1]) == 1)
check("everything else is kept",
      lambda: len(quarantine(_poisoned(), "ledger", "PMT-1005")[0])
              == len(_poisoned()) - 1)
check("re-deciding on the survivors no longer says 'no action'",
      lambda: recheck(quarantine(_poisoned(), "ledger", "PMT-1005")[0]) != "no action required",
      "that is the recovery: drop the source, re-decide, do not argue with the conclusion")
check("with no evidence left it escalates rather than guessing",
      lambda: recheck([]) == "no evidence; escalate",
      "an agent with nothing to go on must say so -- silence is not agreement")

## Run it &mdash; the comparison

In [ ]:
def _compare():
    print("=== clean run ===")
    clean = run_shared()
    for f in clean["findings"]:
        print("  " + str(f)[:110])
    print("  VERDICT:", clean["verdict"])

    print("\n=== one wrong finding from the ledger agent ===")
    bad = run_shared(faulty=True)
    for f in bad["findings"]:
        print("  " + str(f)[:110])
    print("  VERDICT:", bad["verdict"], "   <-- wrong, and nothing reported an error")

    print("\n=== after quarantining the bad source ===")
    kept, dropped = quarantine(bad["findings"], "ledger", "PMT-1005")
    for f in dropped:
        print("  DROPPED " + str(f)[:100])
    print("  VERDICT:", recheck(kept))
guard(_compare)

In [ ]:
if llm_ready():
    # The ONLY difference between these two is the last sentence of the first one.
    LICENSED = ("You are a payments control reviewer. Decide what must happen next, using ONLY "
                "the evidence below. If the evidence is inconsistent or insufficient, say so "
                "instead of deciding.")
    PLAIN    = ("You are a payments control reviewer. Decide what must happen next, using ONLY "
                "the evidence below.")

    def _flags_a_problem(text: str) -> bool:
        return any(w in text.lower() for w in ("inconsist", "insufficient", "cannot determine",
                                               "not enough", "unclear", "contradict"))

    def _model_critic():
        out = run_shared(faulty=True)
        evidence = "\n".join(str(f) for f in out["findings"])
        print("the poisoned evidence a model critic is given:")
        for f in out["findings"]:
            print("  " + str(f)[:110])
        print()
        for label, sysmsg in (("licensed to refuse", LICENSED), ("not licensed", PLAIN)):
            flagged, first = 0, None
            for i in range(3):
                verdict = ask(evidence, system=sysmsg)
                flagged += _flags_a_problem(verdict)
                if i == 0:
                    first = verdict
            print(f"--- {label}: flagged a problem {flagged}/3 ---")
            print("  " + first.strip()[:300] + "\n")
    guard(_model_critic)

### Read it

**The rule-based critic was fooled.** It was handed a well-formed, correctly attributed finding
that happened to be false, and it had no way to know. That is what makes context poisoning
different from an ordinary bug: no exception, no anomaly in the trace, just a confident answer
built on one bad input.

**The model critic depends entirely on one sentence.** The two runs above differ only in whether
the reviewer was told *"if the evidence is inconsistent or insufficient, say so instead of
deciding."* With that licence it reliably notices that a settled payment needs no next action and
says the evidence is inconsistent. Without it, it does what it was asked &mdash; decides &mdash; and
closes the case.

Sit with that for a moment, because it is the most portable thing in this module. The model was
capable of catching the poisoning the whole time. What it lacked was **permission to refuse**. An
agent given only "decide" will decide, on whatever it has, every time. If you have not written
down what it should do when the evidence does not support an answer, you have written an agent
that cannot do anything but answer.

Three more things follow, and they are the take-home of Module 3.

1. **Provenance is not verification.** Every finding in the poisoned run passed `trustworthy()`.
   Knowing where a claim came from does not tell you it is right &mdash; it tells you *what else to
   throw away* when you discover it is wrong. That is Section 4, and it is worth a great deal, but
   it is a recovery mechanism, not a preventative one.
2. **Scope limits the blast radius.** An agent's uncertain working notes belong in `scratch`,
   where they can be inspected and not believed. The shared `findings` list is the thing every
   other agent treats as fact, so the bar for entering it should be higher than "an agent said so".
3. **Shared state is still the right answer.** Lab 1.4 measured what happens without it: agents
   that cannot see each other's work produce worse answers than one agent that can. The fix for
   poisoning is not to go back to isolation &mdash; it is provenance, scope, and a critic that is
   allowed to say the evidence is inconsistent. You have now measured all three.

**What you take from Module 3:** memory that survives a long session; observations the model does
not have to decode; a `StateGraph` you can test without a model in it; checkpoints that make
resume, approval, rewind and audit possible; and a state design that contains one agent's mistake
instead of spreading it. Module 5 puts several agents into exactly this shape.

In [ ]:
score()

## Your turn

1. Give the *rule-based* critic the same licence the model one has: when the findings contain a
   "settled" claim alongside a reason code, or no policy at all, return "evidence inconsistent;
   escalate". Then poison the run and confirm it fires. Which is safer for your organisation: a
   wrong answer or a refusal?
2. Add a `confidence: float` field to `Finding` and have `publish` withhold anything below a
   threshold. Then find the flaw in that idea &mdash; the poisoned finding in this lab would have
   been published with confidence 1.0.
3. Rebuild Section 2's graph with a checkpointer from Lab 3.4, poison it, then use
   `get_state_history()` to find the exact checkpoint at which the bad finding entered. That is
   the incident investigation, and it takes about four lines.